# Attendance shortfall lab
An independently synthetic ML experiment and exact recovery calculator. **No real student data or real-derived statistics are included.** All course names, counts, distributions and future patterns are invented demo assumptions. This public version differs from the private classroom notebook. It predicts simulated final attendance below an example75% theory threshold, not actual college outcomes. Practical/medical/duty-leave policy is not inferred.

In [ ]:
import sys, subprocess, importlib.util
if importlib.util.find_spec("sklearn") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "scikit-learn"])
import io, json, math, zipfile
from pathlib import Path
from fractions import Fraction
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.model_selection import train_test_split, GroupShuffleSplit, GroupKFold, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, ConfusionMatrixDisplay,
    PrecisionRecallDisplay, classification_report)
SEED = 42
OUT = Path("outputs"); OUT.mkdir(exist_ok=True)
plt.rcParams.update({"figure.figsize": (10, 4.5), "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 10})


In [ ]:
SUBJECT_HELD={"COURSE_A":20,"COURSE_B":18,"COURSE_C":16,"COURSE_D":20}
subjects=list(SUBJECT_HELD)
# Independent illustrative parameters, NOT fitted to any student data.
subject_params={"COURSE_A":{"mean":0.80,"spread":0.10},
                "COURSE_B":{"mean":0.72,"spread":0.10},
                "COURSE_C":{"mean":0.85,"spread":0.10},
                "COURSE_D":{"mean":0.68,"spread":0.10}}
GENERATOR_VERSION="independent-synthetic-v1"
def generate_synthetic(n_students=600,seed=42,future_shift=0,cohort="stable"):
    rng=np.random.default_rng(seed);rows=[]
    for i in range(n_students):
        shared=rng.normal(0,0.08)
        for subject,par in subject_params.items():
            q=np.clip(par["mean"]+shared+rng.normal(0,par["spread"]*0.5),0.02,0.98)
            h=SUBJECT_HELD[subject];a=int(rng.binomial(h,q));fh=2*h
            fa=int(rng.binomial(fh,np.clip(q+future_shift,0.02,0.98)))
            rows.append({"synthetic_student_code":f"SIM-{cohort}-{i:05d}","subject":subject,
             "observed_held":h,"observed_attended":a,"future_held":fh,"future_attended":fa,
             "target_shortfall":int(4*(a+fa)<3*(h+fh)),"provenance":"independently_synthetic",
             "generator_version":GENERATOR_VERSION,"random_seed":seed,"synthetic_cohort_id":cohort})
    result=pd.DataFrame(rows);result["observed_rate"]=result.observed_attended/result.observed_held
    result["observed_absent"]=result.observed_held-result.observed_attended
    result["student_mean_observed_rate"]=result.groupby("synthetic_student_code").observed_rate.transform("mean")
    result["student_subjects_below_75"]=result.groupby("synthetic_student_code").observed_rate.transform(lambda x:(x<0.75).sum())
    return result
synth=generate_synthetic()
synth.to_csv(OUT/"synthetic_semesters.csv",index=False)
print("All",len(synth),"records are independently simulated.")


## EDA and task
Features use only the first simulated block; target uses the final simulated attendance total. Future counts and IDs are never predictors. The simulator assumes shared student propensity and independent binomial counts conditional on it. Three blocks form the illustrative semester. No day-level histories or causal claims are made.

In [ ]:
display(synth.groupby("subject").agg(mean_initial_rate=("observed_rate","mean"),shortfall_fraction=("target_shortfall","mean")))
fig,ax=plt.subplots(figsize=(8,4));synth.observed_rate.hist(ax=ax,bins=15,color="#73a89a")
ax.set(title="Independent synthetic first-block attendance",xlabel="Attendance fraction",ylabel="Simulated student-subject rows")
plt.tight_layout();plt.savefig(OUT/"synthetic_eda.png",dpi=140);plt.show()


## Training-only feature selection and model comparison
Students are disjoint across development/validation/test. SelectKBest and preprocessing are fitted inside group-aware cross-validation. Compare Logistic Regression and shallow Decision Tree, with majority and current-rate baselines. Choose model and alert threshold on validation only.

In [ ]:
minimal=["subject","observed_rate","observed_held"]
expanded=minimal+["observed_absent","student_mean_observed_rate","student_subjects_below_75"]
trainval_ix,test_ix=next(GroupShuffleSplit(n_splits=1,test_size=0.2,random_state=SEED).split(synth,groups=synth.synthetic_student_code))
dev=synth.iloc[trainval_ix].copy(); test=synth.iloc[test_ix].copy()
tr_ix,va_ix=next(GroupShuffleSplit(n_splits=1,test_size=0.25,random_state=SEED+1).split(dev,groups=dev.synthetic_student_code))
train=dev.iloc[tr_ix].copy(); val=dev.iloc[va_ix].copy()
assert not (set(train.synthetic_student_code)&set(val.synthetic_student_code))
assert not (set(dev.synthetic_student_code)&set(test.synthetic_student_code))
for name,part in [("train",train),("validation",val),("test",test)]:
    print(name,len(part),"synthetic rows; positive fraction",round(part.target_shortfall.mean(),3))

def build_pipeline(features,model):
    numeric=[f for f in features if f!="subject"]
    numeric_pipeline=Pipeline([("impute",SimpleImputer(strategy="median")),("scale",StandardScaler())])
    prep=ColumnTransformer([("numeric",numeric_pipeline,numeric),
        ("subject",OneHotEncoder(handle_unknown="ignore",sparse_output=False),["subject"])])
    return Pipeline([("prepare",prep),("select",SelectKBest(f_classif,k="all")),("model",model)])

def metrics(y,prob,pred):
    result={"accuracy":accuracy_score(y,pred),"precision":precision_score(y,pred,zero_division=0),
            "recall":recall_score(y,pred,zero_division=0),"f1":f1_score(y,pred,zero_division=0)}
    result["roc_auc"]=roc_auc_score(y,prob) if len(np.unique(y))==2 else np.nan
    result["average_precision"]=average_precision_score(y,prob) if len(np.unique(y))==2 else np.nan
    return result

candidates={}; validation=[]
for feature_name,features in [("subject_only",minimal),("cross_subject",expanded)]:
    for model_name,model,grid in [
        ("LR",LogisticRegression(max_iter=1500,random_state=SEED),{"model__C":[0.1,1],"model__class_weight":[None,"balanced"]}),
        ("Tree",DecisionTreeClassifier(random_state=SEED),{"model__max_depth":[3,5],"model__min_samples_leaf":[20],"model__class_weight":[None,"balanced"]})]:
        search=GridSearchCV(build_pipeline(features,model),
            {**grid,"select__k":[5,"all"]},scoring="average_precision",cv=GroupKFold(3),n_jobs=1)
        search.fit(train[features],train.target_shortfall,groups=train.synthetic_student_code)
        key=feature_name+"_"+model_name;candidates[key]=(search.best_estimator_,features)
        prob=search.predict_proba(val[features])[:,1]
        validation.append({"model":key,**metrics(val.target_shortfall,prob,prob>=0.5),
                           "cv_ap":search.best_score_,"settings":str(search.best_params_)})
validation=pd.DataFrame(validation).sort_values("average_precision",ascending=False)
display(validation)
best_name=validation.iloc[0].model
winner,features=candidates[best_name]
# Choose alert threshold using validation F1, not test labels or an invented institutional threshold.
val_prob=winner.predict_proba(val[features])[:,1]
thresholds=np.linspace(0.1,0.9,33)
threshold=float(max(thresholds,key=lambda t:f1_score(val.target_shortfall,val_prob>=t,zero_division=0)))
print("Selected on synthetic validation:",best_name,"alert score threshold",threshold)
print("Score is not a guaranteed calibrated probability; this threshold is not college policy.")


## Synthetic test benchmark
All metrics measure the simulator, not real-world prediction. Precision, recall, F1, ROC-AUC and average precision accompany accuracy. A majority baseline can look strong without being useful. Tree thresholds are standardized values.

In [ ]:
results=[]; fitted={}
for name,(est,fs) in candidates.items():
    est.fit(dev[fs],dev.target_shortfall)
    prob=est.predict_proba(test[fs])[:,1]
    t=threshold if name==best_name else 0.5
    results.append({"model":name,**metrics(test.target_shortfall,prob,prob>=t)})
    fitted[name]=(est,fs)
dummy=DummyClassifier(strategy="most_frequent").fit(dev[minimal],dev.target_shortfall)
prob=dummy.predict_proba(test[minimal])[:,list(dummy.classes_).index(1)]
results.append({"model":"majority baseline",**metrics(test.target_shortfall,prob,dummy.predict(test[minimal]))})
projection_score=1-test.observed_rate.to_numpy()
results.append({"model":"current-rate projection",**metrics(test.target_shortfall,projection_score,test.observed_rate<0.75)})
results=pd.DataFrame(results)
print("SYNTHETIC BENCHMARK ONLY - NOT VALIDATED ON FUTURE AIT ATTENDANCE")
display(results.round(3));results.to_csv(OUT/"synthetic_benchmark_only.csv",index=False)
final_model,features=fitted[best_name];test_prob=final_model.predict_proba(test[features])[:,1]
fig,axes=plt.subplots(1,2,figsize=(11,4.5))
ConfusionMatrixDisplay.from_predictions(test.target_shortfall,test_prob>=threshold,
    display_labels=["Meets75%","Below75%"],ax=axes[0],colorbar=False,cmap="Greens")
axes[0].set_title("Synthetic test confusion matrix")
PrecisionRecallDisplay.from_predictions(test.target_shortfall,test_prob,ax=axes[1],name=best_name)
axes[1].set_title("Synthetic test precision-recall")
fig.tight_layout();fig.savefig(OUT/"synthetic_test_metrics.png",dpi=160);plt.show()
feat_names=final_model.named_steps["prepare"].get_feature_names_out()
selected=feat_names[final_model.named_steps["select"].get_support()]
print("Selected transformed features:",selected.tolist())
# Tree split values below are standardized z-scores, not raw attendance percentages.
tree_model,tree_fs=fitted["subject_only_Tree"]
tree_names=tree_model.named_steps["prepare"].get_feature_names_out()[tree_model.named_steps["select"].get_support()]
fig,ax=plt.subplots(figsize=(16,6))
plot_tree(tree_model.named_steps["model"],feature_names=tree_names.tolist(),
          class_names=["Meets75%","Below75%"],max_depth=2,filled=True,fontsize=8,ax=ax)
ax.set_title("Synthetic Decision Tree: standardized feature values, top levels only")
fig.tight_layout();fig.savefig(OUT/"synthetic_tree.png",dpi=140);plt.show()


## Future-pattern sensitivity
Improving and declining futures are assumed scenarios. They do not create new real evidence.

In [ ]:
sensitivity=[]
for name,shift,seed in [("stable",0,100),("improving",0.10,101),("declining",-0.10,102)]:
    scenario=generate_synthetic(400,seed,shift,name)
    score=final_model.predict_proba(scenario[features])[:,1]
    sensitivity.append({"assumed_future":name,"target_fraction":scenario.target_shortfall.mean(),
                        **metrics(scenario.target_shortfall,score,score>=threshold)})
display(pd.DataFrame(sensitivity).round(3))
pd.DataFrame(sensitivity).to_csv(OUT/"synthetic_future_sensitivity.csv",index=False)


## Interactive recovery calculator
Enter hypothetical or your own counts during your private session. Nothing is uploaded. This is arithmetic, not ML.75% is an example default, not institutional eligibility advice. Unknown excusal, makeup and attendance-credit rules must be confirmed separately.

In [ ]:
import ipywidgets as widgets
try:
 from google.colab import output
 output.enable_custom_widget_manager()
except ImportError: pass
attended=widgets.BoundedIntText(value=12,min=0,max=500,description="Attended")
held=widgets.BoundedIntText(value=20,min=1,max=500,description="Held")
remaining=widgets.BoundedIntText(value=20,min=0,max=500,description="Remaining")
card=widgets.HTML()
def update(change=None):
 a,h,r=attended.value,held.value,remaining.value
 if a>h:card.value="Attended cannot exceed held.";return
 consecutive=max(0,3*h-4*a)
 needed=max(0,-(-(3*(h+r)-4*a)//4))
 feasible=needed<=r
 card.value=f"<h3>Example75% recovery scenario</h3><p>Current: {100*a/h:.1f}%. Consecutive attendances to recover: {consecutive}.</p><p>Need {needed} of {r} remaining sessions. Feasible by attendance alone: {feasible}.</p><small>No exemption or duty-leave adjustments. Not official eligibility.</small>"
for w in [attended,held,remaining]:w.observe(update,names="value")
update();display(widgets.VBox([widgets.HBox([attended,held,remaining]),card]))


## Limits and next steps
This public notebook contains neither a real cohort nor real-derived distribution parameters. It is a reproducible curriculum/portfolio demo, not a validated deployed predictor. For a real study, obtain authorized longitudinal records and a later completed-semester holdout. Never publish classmates' private records, even with names removed. Keep any real-data notebook outside this public repository.

References: https://scikit-learn.org/stable/common_pitfalls.html and https://scikit-learn.org/stable/auto_examples/model_selection/plot_precision_recall.html .